# Paper 2 — Calibrated forecasting, cross-region transfer, robustness, statistics
Runs the four Paper-2 contributions on top of the Paper-1 benchmark, all multi-seed (5 seeds) and resume-safe (re-run after a Colab disconnect continues from per-seed checkpoints):

1. **Cross-region transfer** — pretrain a calibrated HASI-Net on Chicago (data-rich), transfer resolution-agnostic weights to Madhya Pradesh (data-sparse) with an L2-SP region-adaptation regulariser, vs from-scratch.
2. **Sparsity-aware calibrated head** — gated ZINB/NB + monotone quantile head vs the deterministic point head, on both MP and Chicago. Reports CRPS, 80% coverage, sharpness, pinball.
3. **Missing-data robustness** — mask 0/10/25/50% of training cells; intervals should widen, not collapse.
4. **Diebold-Mariano pairwise tests** — HASI-Net vs GraphWaveNet vs HA, h-step corrected.
5. **Friedman + Nemenyi and bootstrap CIs** over the saved Paper-1 per-seed tables.

Every result is written from a real run; nothing is fabricated.

## 0. Environment

In [ ]:
# Install dependencies (Colab GPU runtime).
import subprocess, sys
def pip(pkg): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
for p in ["torch", "numpy", "pandas", "requests", "geopandas", "shapely", "matplotlib", "pyarrow", "scipy"]:
    try: __import__(p)
    except ImportError: pip(p)
print("deps ready")

## 1. Mount Drive and configure

In [ ]:
# Mount Drive and put the repo on the path.
import sys, pathlib, os
from google.colab import drive
drive.mount("/content/drive")

REPO = pathlib.Path("/content/drive/MyDrive/spatio-temporal-crime")
if not (REPO / "src" / "hasi_net").exists():
    REPO = pathlib.Path("/content/spatio-temporal-crime")
if not (REPO / "src" / "hasi_net").exists():
    raise SystemExit("Repo not found. Upload spatio-temporal-crime to MyDrive or clone it into /content.")
sys.path.insert(0, str(REPO / "src"))
os.chdir(str(REPO))
print("Repo:", REPO)

In [ ]:
# Paper-2 configs: calibrated heads, negative-binomial count loss.
# MP has only ~7 training windows (14 years yearly, lookback 4 / horizon 2), so
# we use a conservative lr and epochs to avoid catastrophic forgetting of the
# transferred weights; fine_tune_l2sp further drops transferred-param lr to 0.1x.
from hasi_net.config import Config, MADHYA_PRADESH, RESULTS_DIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 1, 2, 3, 4]

cfg_chi = Config(target_region=MADHYA_PRADESH, use_chicago_benchmark=True,
                 chicago_year_start=2015, chicago_year_end=2024,
                 device="cuda", lookback=12, horizon=3,
                 epochs=80, batch_size=64, lr=1e-3, hidden_dim=64,
                 loss_type="nb", calibrated_head=True, pso_enabled=False)
cfg_mp = Config(target_region=MADHYA_PRADESH, device="cuda",
                lookback=4, horizon=2, epochs=100, batch_size=16,
                lr=5e-4, hidden_dim=64, loss_type="nb",
                calibrated_head=True, pso_enabled=False, patience=15)
print("Chicago config:", cfg_chi.to_dict())
print("MP config:", cfg_mp.to_dict())

## 2. Build both panels (cached after first run)

In [ ]:
from hasi_net.data import build_chicago_panel, build_mp_crime_panel
chi_panel = build_chicago_panel(cfg_chi)
mp_panel = build_mp_crime_panel(cfg_mp)
print("Chicago [T,N,C]:", chi_panel.counts.shape, chi_panel.categories)
print("MP [T,N,C]:", mp_panel.counts.shape, mp_panel.categories)

## 3. Cross-region transfer (Chicago -> MP)

In [ ]:
from hasi_net.transfer import run_transfer_vs_scratch
tr = run_transfer_vs_scratch(cfg_chi, cfg_mp, seeds=SEEDS, lam=1e-3,
                             tag="p2", verbose=True)
tr.round(4)

## 4. Calibrated vs point head (MP, then Chicago)

In [ ]:
from hasi_net.p2_experiments import run_calibrated_vs_point
run_calibrated_vs_point("mp", cfg_mp, seeds=SEEDS, tag="p2", verbose=True)
run_calibrated_vs_point("chicago", cfg_chi, seeds=SEEDS, tag="p2", verbose=True)

## 5. Missing-data robustness (MP)

In [ ]:
from hasi_net.p2_experiments import run_missing_data_robustness
run_missing_data_robustness("mp", cfg_mp, seeds=SEEDS,
                            fractions=[0.0, 0.1, 0.25, 0.5], tag="p2", verbose=True)

## 6. Diebold-Mariano pairwise tests (Chicago, point head, h=3)

In [ ]:
from hasi_net.p2_experiments import run_dm_comparison
run_dm_comparison("chicago",
                  cfg_chi.override(calibrated_head=False, loss_type="log1p_mse"),
                  seeds=SEEDS,
                  pairs=[("HASI-Net", "GraphWaveNet"),
                         ("HASI-Net", "HA"),
                         ("GraphWaveNet", "HA")],
                  tag="p1", h_step=3, verbose=True)

## 7. Friedman + Nemenyi and bootstrap CIs over the Paper-1 tables
Reads `results/metrics_p1_chicago_h*_perseed.csv`. Run **Paper 1 first** (notebook `p1_multiseed_colab.ipynb`); this step skips gracefully if those files are absent.

In [ ]:
from hasi_net.stats import friedman_nemenyi, bootstrap_ci
import numpy as np, pandas as pd
models = ["HA","LSTM","STGCN","GraphWaveNet","DCRNN","MTGNN","InformerOnly","HASI-Net"]
blocks = []
for h in (3,6,12):
    f = RESULTS_DIR / f"metrics_p1_chicago_h{h}_perseed.csv"
    if not f.exists(): continue
    df = pd.read_csv(f)
    for s in SEEDS:
        vals = []
        for m in models:
            sel = df[(df.model==m)&(df.seed==s)]["MAE"].values
            vals.append(float(sel[0]) if len(sel) else np.nan)
        if not np.isnan(vals).any(): blocks.append(vals)
if len(blocks) >= 2:
    perf = np.array(blocks)
    fr = friedman_nemenyi(perf, alpha=0.1)
    print(f"Friedman chi2={fr['chi2']:.3f} p={fr['p']:.4g} n={fr['n_blocks']} k={fr['k']} CD={fr['cd']:.3f}")
    for m, r in sorted(zip(models, fr["mean_ranks"]), key=lambda x: x[1]):
        _, lo, hi = bootstrap_ci(perf[:, models.index(m)], 0.95, seed=0)
        print(f"  {m:14s} rank {r:.2f}  MAE_CI [{lo:.4f},{hi:.4f}]")
else:
    print("Paper-1 per-seed tables not found -- run p1_multiseed_colab.ipynb first.")

## 8. Persist / verify P2 artifacts

In [ ]:
import pathlib
res = pathlib.Path(REPO) / "results"
arts = sorted(p.name for p in res.glob("*p2*")) + sorted(p.name for p in res.glob("dm_*p1*"))
print("P2 artifacts:")
for a in arts: print(" ", a)